# tailbench — run analysis

Charts one run directory. To compare runs, use `compare_runs.ipynb`.

```bash
python3 -m venv .venv
.venv/bin/pip install pandas numpy matplotlib jupyterlab ipykernel
.venv/bin/jupyter lab notebooks/analyse_run.ipynb
```

Paths below assume the kernel starts in `notebooks/`; drop the `../` if it starts
at the repo root.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd()/"tailbench_viz.py").exists()
                      else pathlib.Path.cwd()/"notebooks"))

import matplotlib.pyplot as plt
import pandas as pd
import tailbench_viz as tv

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.titlesize": 11, "figure.facecolor": "white"})
pd.set_option("display.width", 200, "display.max_columns", 50)

## 1. Pick a run

Run directories are timestamp-prefixed, so the last is the most recent.

In [ ]:
available = tv.load_runs("../results")
tv.compare(available)

In [ ]:
RUN = available[-1].path          # or: "../results/20260825-012635-fanout-bimodal-001"
run = tv.load_run(RUN)

print(f"{run.label}   scenario={run.scenario_id}   seed={run.manifest['seed']}")
print(f"budget {run.budget_ms:g} ms   penalty {run.penalty_ms:g} ms   "
      f"warmup {run.warmup_s:g}s   n={len(run.df):,} ({int(run.df.post_warmup.sum()):,} post-warmup)")
print(f"environment={run.manifest['environment']}"
      + ("" if run.authoritative else "   <- NON-AUTHORITATIVE, do not quote as a result"))
if run.report.get("failed"):
    print(f"RUN FAILED: {run.report['failure_reason']}")

## 2. The whole run on one page

All eight panels at once. The sections below take them one at a time.

In [ ]:
tv.overview(run);

## 3. Outcomes

What fraction of requests succeeded. Read this before any latency number: a tail
statistic means nothing until you know what it was computed over.

Nonzero `incorrect` means the run skipped required calls or the digest did not
match — no latency number from it counts.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6))
tv.plot_outcomes(run, ax)

d = run.df[run.df.post_warmup]
print(d.outcome.value_counts().to_frame("count").assign(
    pct=lambda t: (100 * t["count"] / len(d)).round(4)).to_string())

## 4. Latency distribution and tail

**Left:** where requests land, with the metric landmarks drawn on. The pile at the
right edge is failed requests entering at `penalty_ms` — that is what stops failure
from beating slowness.

**Right:** the tail, plotted as `1 - F` on a log axis so p99 and p99.9 are both
legible. Both charts are log-scaled because the tail sits orders of magnitude past
the median.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
tv.plot_latency_dist(run, axes[0])
tv.plot_tail_cdf(run, axes[1])
fig.tight_layout()

### Percentile vs CVaR

`p99` is one order statistic: flat in `penalty_ms` below 1% failures, equal to it
above. `cvar_99` averages the worst 1%, so it responds throughout. The gap between
the columns is what `p99` cannot see.

The `scored` vs `ok-only` gap below is the attrition premium: how much of the tail
is failed requests rather than slow ones.

In [ ]:
s = run.scored()
display(pd.DataFrame([
    {"q": f"{q*100:g}%", "percentile_ms": tv.percentile(s, q), "cvar_ms": tv.cvar(s, q)}
    for q in (0.5, 0.9, 0.99, 0.999)
]).set_index("q").round(3))

ok = run.df[(run.df.post_warmup) & (run.df.outcome == "ok")].e2e_ms
print(f"p99      scored {tv.percentile(s, .99):8.2f} ms   ok-only {tv.percentile(ok, .99):8.2f} ms")
print(f"cvar_99  scored {tv.cvar(s, .99):8.2f} ms   ok-only {tv.cvar(ok, .99):8.2f} ms")

## 5. Over time

**Top:** rolling percentiles, with non-`ok` counts shaded behind. A *rising* p99
means a queue is building; *flat with spikes* means the tail is arrival-driven.
Different causes, different fixes, and the aggregate cannot tell them apart.

**Bottom:** offered load against completed-ok. A gap between them is overload.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7.5))
tv.plot_latency_over_time(run, axes[0], window_s=1.0, quantiles=(0.5, 0.9, 0.99))
tv.plot_throughput(run, axes[1], window_s=1.0)
fig.tight_layout()

## 6. Where the tail comes from

**Left, by class:** classes differ in `requires`, so they differ in exposure. If a
10%-weight class owns the tail, the aggregate hides which fan-out to fix.

**Right, by downstream:** service time is fixed by the scenario; queue wait is the
program's own doing. A tall queue bar next to a short service bar is a self-inflicted
bottleneck — the most actionable signal here, and invisible end-to-end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
tv.plot_by_class(run, axes[0])
tv.plot_downstreams(run, axes[1])
fig.tight_layout()

In [ ]:
sp = run.spans()
sp = sp[sp.post_warmup]
if sp.empty:
    print("no downstream calls recorded")
else:
    summary = sp.groupby("downstream_id").apply(lambda g: pd.Series({
        "calls": len(g),
        "queue_p50": tv.percentile(g.queue_wait_ms, .50),
        "queue_p99": tv.percentile(g.queue_wait_ms, .99),
        "service_p50": tv.percentile(g.service_ms, .50),
        "service_p99": tv.percentile(g.service_ms, .99),
        "total_p99": tv.percentile(g.total_ms, .99),
        "queue_share_of_p99_%": 100 * tv.percentile(g.queue_wait_ms, .99)
                                 / max(tv.percentile(g.total_ms, .99), 1e-9),
        "timeouts": int((g.call_outcome == "timeout").sum()),
    }), include_groups=False).round(3)
    display(summary)

    # `requires` is a minimum, so >1 call per required downstream is a deliberate
    # retry or hedge -- trading downstream load for tail latency.
    per_req = sp.groupby("request_id").size()
    print(f"calls per request: mean {per_req.mean():.2f}, max {per_req.max()}")
    if (sp.attempt > 0).any():
        print(f"retries/hedges: {int((sp.attempt > 0).sum()):,} calls with attempt > 0")
    else:
        print("no retries or hedging (every call was attempt 0)")

## 7. Can the measurement be trusted?

Not a property of the program. If the generator dispatched late, part of the measured
latency is harness lag and the tail is *optimistic*. Gate: 0.1% of requests over 1ms.

Anything other than `linux-pinned` is non-authoritative. On macOS the ~1ms timer floor
trips this routinely — such runs are fine for relative comparison at a fixed seed, but
are not quotable as absolute results.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.4))
tv.plot_dispatch_health(run, ax)

d = run.df[run.df.post_warmup]
late = d[d.late_dispatch_ms > 1.0]
print(f"late dispatches   {len(late):,} / {len(d):,} = {100*len(late)/max(len(d),1):.3f}%  (gate 0.1%)")
if len(late):
    print(f"  median late by  {late.late_dispatch_ms.median():.3f} ms")
    print(f"  p99 late by     {tv.percentile(late.late_dispatch_ms, .99):.3f} ms")
    print(f"  max late by     {late.late_dispatch_ms.max():.3f} ms")
    # Rising median => generator falling behind. Flat => timer jitter.
    first, second = late.iloc[:len(late)//2], late.iloc[len(late)//2:]
    print(f"  1st half median {first.late_dispatch_ms.median():.3f} ms, "
          f"2nd half {second.late_dispatch_ms.median():.3f} ms"
          "   (rising => falling behind; flat => jitter)")
    print(f"  of {len(late):,} late-dispatched, {len(late[late.outcome != 'ok']):,} also failed")
print(f"\nenvironment: {run.manifest['environment']}"
      + ("  (authoritative)" if run.authoritative else "  <- relative comparison only"))

---

Next: **`compare_runs.ipynb`** — baseline vs candidate, which is where a change to
`program.rs` is judged.